In [1]:
import pandas as pd
import json
import numpy as np
import torch
import torch.nn as nn
import re
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BOS, EOS = ' ', '\n'
PAD = "<PAD>"
with open('/content/drive/MyDrive/Colab Notebooks/songs/P1harmony.json', 'r') as file:
    text = json.load(file)
verses = re.findall(r'Verse.*?(?=Chorus|Verse|\Z)', text, re.DOTALL)
choruses = re.findall(r'Chorus.*?(?=Verse|Chorus|\Z)', text, re.DOTALL)

verses = [re.sub(r'\[.*?\]', '', couplet).strip() for couplet in verses]
choruses= [re.sub(r'\[.*?\]', '', chorus).strip() for chorus in choruses]

lines = ' '.join(verses + choruses)

In [ ]:
tokens = set(" ".join(lines))

tokens = sorted(tokens)
n_tokens = len(tokens)
print ('n_tokens = ',n_tokens)

n_tokens =  2455


In [ ]:
token_to_id = {symbol: index for index, symbol in enumerate(tokens, start=1)}
token_to_id[PAD] = 0


n_tokens = len(token_to_id)
n_tokens

2456

In [ ]:
class SymbolDataset(Dataset):

    def init(self, data, token_to_ids):
        self.data = data
        self.token_to_ids = token_to_ids

    def len(self):
        return len(self.data)

    def getitem(self, index):
        symbols = [self.token_to_ids[symbol] for symbol in self.data[index]]
        symbols.append(self.token_to_ids['EOS'])
        return symbols

In [ ]:
train_lines, dev_lines = train_test_split(lines, test_size=0.25, random_state=42)
train_dataset = SymbolDataset(train_lines, token_to_id)
train_loader = DataLoader(train_dataset, batch_size=64)
def collation(batch, pad=token_to_id[PAD], dtype=np.int64):
    max_len = max(map(len, batch))
    lines_ix = np.full([len(batch), max_len], pad, dtype=dtype)
    for i in range(len(batch)):
        lines_ix[i, :len(batch[i])] = batch[i]
    return torch.from_numpy(lines_ix)


train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collation)
test_dataset = SymbolDataset(dev_lines, token_to_id)
test_loader = DataLoader(test_dataset, batch_size=64, collate_fn=collation)

In [ ]:
class LSTM(nn.Module):
    def __init__(self, n_vocab=2458, hidden_dim=256, embedding_dim=256):
        super(LSTM, self).__init__()

        self.hidden_dim = hidden_dim
        self.embedding_dim = embedding_dim
        self.lstm = nn.LSTM(embedding_dim, hidden_dim)
        self.embeddings = nn.Embedding(n_vocab, embedding_dim)
        self.fc = nn.Linear(hidden_dim, n_vocab)

    def forward(self, seq):
        embedded = self.embeddings(seq.t())
        lstm_out, _ = self.lstm(embedded)
        ht = lstm_out[-1]
        out = self.fc(ht)
        return out

In [ ]:
PAD = "<PAD>"
token_to_id[PAD]=0
class Window(nn.Module):
    def __init__(self, n_tokens=2458, emb_size=256, hid_size=256):
        super().__init__()
        PAD = "<PAD>"
        stride = 1
        kernel_size = 5
        num_leading_zeros = (kernel_size - 1) * stride
        self.embedding = nn.Embedding(n_tokens, emb_size,  padding_idx=token_to_id[PAD])
        self.padding = nn.ZeroPad2d((num_leading_zeros, 0, 0, 0))
        self.conv = nn.Conv1d(emb_size, hid_size, kernel_size=kernel_size, stride=stride)
        self.lr = nn.Linear(hid_size, n_tokens)

    def __call__(self, input_ix):
        emb = self.embedding(input_ix)
        emb = emb.permute(0, 2, 1)
        padded = self.padding(emb)
        conved = self.conv(padded)
        relu = F.relu(conved)
        relu = relu.permute(0, 2, 1)
        out = self.lr(relu)
        return out

In [ ]:
model2= LSTM()
model2.cuda()
model3 = Window()
model3.cuda()

Window(
  (embedding): Embedding(2458, 256, padding_idx=0)
  (padding): ZeroPad2d((4, 0, 0, 0))
  (conv): Conv1d(256, 256, kernel_size=(5,), stride=(1,))
  (lr): Linear(in_features=256, out_features=2458, bias=True)
)

In [ ]:
n_epochs = 10
optimizer = torch.optim.Adam(model2.parameters(), lr=1e-3)
for epoch in range(n_epochs):
    model2.train()
    loss_fn = torch.nn.CrossEntropyLoss()
    avg_loss = 0.
    for i, (x_batch, y_batch) in enumerate(train_loader):
        y_pred = model2(x_batch)

        loss = loss_fn(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()

        optimizer.step()
        avg_loss += loss.item() / len(train_loader)
    print('Epoch {}/{} \t loss={:.7f}'.format(epoch + 1, n_epochs, avg_loss))

    total_batches = 0.0
    test_loss = 0.0
    test_perplexity = 0.0
    model2.eval()
    for i, (x_batch, y_batch) in enumerate(test_loader):
        with torch.no_grad():
            y_pred = model2(x_batch)
            loss = loss_fn(y_pred, y_batch)
            test_loss += loss.item()
            total_batches += 1

Epoch 1/10 	 loss=2.4062777
Epoch 2/10 	 loss=1.8748338
Epoch 3/10 	 loss=1.6462575
Epoch 4/10 	 loss=1.4903131
Epoch 5/10 	 loss=1.3712301
Epoch 6/10 	 loss=1.2778401
Epoch 7/10 	 loss=1.2018645
Epoch 8/10 	 loss=1.1413467
Epoch 9/10 	 loss=1.0909909
Epoch 10/10 	 loss=1.0503569


In [ ]:
opt = torch.optim.Adam(model3.parameters(), lr=1e-3)
for epoch in range(10):
    model3.train()
    for i, batch in train_loader:
        x, y = batch[:, :-1], batch[:, 1:]
        out = model3(x.cuda())
        out = out.view(out.shape[0] * out.shape[1], -1)
        y = y.cuda().flatten()
        loss_i = F.cross_entropy(out, y, ignore_index=token_to_id[PAD])
        opt.zero_grad()
        loss_i.backward()
        opt.step()
        total_loss += loss_i.item()
        total_batches += 1

    avg_loss = total_loss / total_batches

    print('Epoch {}/{} \t loss={:.7f}'.format(epoch + 1, n_epochs, avg_loss))

    model3.eval()
    total_loss = 0
    total_batches = 0
    with torch.no_grad():
        for i, batch in test_loader:
            x, y = batch[:, :-1], batch[:, 1:]
            out = model3(x.cuda())
            out = out.view(out.shape[0] * out.shape[1], -1)
            y = y.cuda().flatten()
            loss_i = F.cross_entropy(out, y, ignore_index=token_to_id[PAD])
            total_loss += loss_i.item()
            total_batches += 1

Epoch 1/10 	 loss=2.8837494
Epoch 2/10 	 loss=2.2866299
Epoch 3/10 	 loss=2.0717482
Epoch 4/10 	 loss=1.9348050
Epoch 5/10 	 loss=1.8366334
Epoch 6/10 	 loss=1.7585917
Epoch 7/10 	 loss=1.6967275
Epoch 8/10 	 loss=1.6450461
Epoch 9/10 	 loss=1.5978666
Epoch 10/10 	 loss=1.5589671


In [ ]:
id_to_token = {i: symbol for symbol, i in token_to_id.items()}
def generate(model, prompt, max_len=250, temperature=0):
    model.eval()
    token_ids = [token_to_id[symbol] for symbol in prompt]
    input_tensor = torch.LongTensor(token_ids).unsqueeze(0).cuda()
    generated_symbols = []

    criterion = torch.nn.CrossEntropyLoss()

    total_prob = 0.0
    num_tokens = 0

    while len(generated_symbols) < max_len:
        with torch.no_grad():
            out = model(input_tensor)[:, -1, :]
            if temperature == 0:
                max_v, ind = torch.max(out, axis=1)
                token = ind.cpu()[0].item()
            else:
                probs = F.softmax(out / temperature, dim=1)
                token = np.random.choice(list(range(len(token_to_id))), p=probs[0].cpu().numpy())
                ind = torch.LongTensor([token]).cuda()

            if token == token_to_id[EOS]:
                break

            loss = criterion(out, torch.LongTensor([token]).cuda())
            probs += loss.item()
            num_tokens += 1

            generated_symbols.append(id_to_token[token])
            input_tensor = torch.cat([input_tensor, ind.unsqueeze(0)], axis=1)

    perplexity = torch.exp(probs / num_tokens)
    generated_text = prompt + "".join(generated_symbols)

    return perplexity, generated_text

In [ ]:
word = '바'

In [ ]:
perplexity, text = generate(model2, word, temperature=0.2)
text

바 가면 (가면 가면) 열정페이 (Ay)
학교 가면 (가면 가면) 선생님 (Shit, woo)
상사들은 (No, woo, woo) 행패 (Yeah, yeah)
언론에선 맨날 몇 포 세대 (Woo, bang)
  
  
  
룰 바꿔 change change (Woo)
황새들은 원해 원해 maintain
그렇게는 안 되지 BANG BANG
이건 정상이 아냐
이건 정상이 아냐
  
  
아 노력노력 타령 좀 그만둬
아 오그라들어 내 두 손발도
아 노력 노력 아 노력 노력
아 노랗구나 싹수가 (역시 황새!)
노력타령 좀 그만둬
아 오그라들어 내 두 손발도
아 노력 노력 아 노력 노력
아 노랗구나 싹수가
  
  
(역시 황새야) 실망 안 시켜
(역시 황새야) 이름 값 하네
(역시 황새야) 다 해먹어라
(역시 황새야) 황새야
They call me (Call me) 뱁새 (뱁새)
욕봤지 이 세대 (이 세대)
빨리 (Woo) chase m hase them
황새 덕에 내 가랑인 탱탱
  
  
  
So call me 뱁새 (뱁새)
욕봤지 이 세대 (세대)
빨리 (Woo) chase hase them
금수저로 태어난 내 선생님)


In [ ]:
perplexity

2.0717482267267

In [ ]:
perplexity, text = generate(model3, word, temperature=0.1)
perplexity

5.6967275862782